# Pull Measurements from Annotations (not predictions)
This acts as a temporary script to do example ecological application work. It will pull the original masks instead of the predicted masks to generate measurements of TL and body span. 

In [1]:
import os
import json
import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask

from offline_eval_functions import *

# Filter Annotations to only those with Valid Segmentation Mask

✅ No overlapping flight-date pairs found.


# Generate Segmentation Mask .png

In [2]:
def create_masks_from_coco(json_file, output_dir):
    # Load the annotations using COCO
    coco = COCO(json_file)

    # Get all image ids in the dataset
    image_ids = coco.getImgIds()
    
    os.makedirs(output_dir, exist_ok=True)  # Ensure output directory exists

    for image_id in image_ids:
        # Get the annotations for the image
        ann_ids = coco.getAnnIds(imgIds=image_id)
        annotations = coco.loadAnns(ann_ids)

        # Get the image info
        img_info = coco.loadImgs(image_id)[0]
        height, width = img_info['height'], img_info['width']
        file_name = img_info['file_name']  # Extract the filename

        # Create an empty mask
        combined_mask = np.zeros((height, width), dtype=np.uint8)

        for annotation in annotations:
            if 'segmentation' in annotation:
                rle = coco_mask.frPyObjects(annotation['segmentation'], height, width)
                
                # Decode the mask
                mask = coco_mask.decode(rle)
                
                # Combine the mask into a single mask
                combined_mask = np.maximum(combined_mask, mask)

        # Save the combined mask as PNG with the image filename appended with _annotation
        mask_image_name = os.path.splitext(file_name)[0] + '_annotation.png'
        mask_image = Image.fromarray(combined_mask * 255)  # Scale to 0-255
        mask_image.save(os.path.join(output_dir, mask_image_name))

# Example usage
json_file = '/mnt/class_data/group2/alexandradigiacomo/dataset/annotations/crop_center_coord/test.json'  # Path to your JSON file
output_dir = '/mnt/class_data/group2/alexandradigiacomo/dataset/annotations/crop_center_coord/all_segmentation_masks'  # Directory to save masks
create_masks_from_coco(json_file, output_dir)

loading annotations into memory...
Done (t=0.08s)
creating index...
index created!


# Merge with Ground Truth 

In [3]:
# metadata
metadata = '/mnt/class_data/group2/alexandradigiacomo/dataset/metadata/metadata.csv'
metadata_lengths = '/mnt/class_data/group2/alexandradigiacomo/dataset/metadata/sharklengths_metadata.csv' # photog lengths

# make dataframes - double merging some columns; to be cleaned
df_meta = pd.read_csv(metadata) # metadata from exif
df_meta.rename(columns={'FileName': 'filename'}, inplace=True)

df_meta_len = pd.read_csv(metadata_lengths) # measurements (photogrammetry)
df_meta_len.rename(columns={'FileName': 'filename'}, inplace=True)

# merge
df_meta_full = pd.merge(df_meta_len, df_meta, on=['filename'], how='left')

In [23]:
root_pngs = '/mnt/class_data/group2/alexandradigiacomo/dataset/annotations/crop_center_coord/all_segmentation_masks'
annot_files = [f for f in sorted(os.listdir(root_pngs))] # list of annotation .pngs

ValueError: max() iterable argument is empty

In [37]:
def process_biometrics(root_predictions, pred_files): 
    """Compute morphometric variables from input pred_files and return a dataframe."""
    data = []  # Store rows for the dataframe
    for file in pred_files:
        mask_path = os.path.join(root_predictions, file)  # Full path to the mask
        mask_raw = Image.open(mask_path)
        mask = np.array(mask_raw)

        # Check if the mask is blank (all zeros)
        if np.count_nonzero(mask) == 0:
            #print(f"Skipping {file}: Mask is blank (all zeros).")
            continue  # Skip this file if the mask is blank

        # Ensure the mask is binary (0 or 1)
        mask = (mask > 0).astype(np.uint8)  # Convert all non-zero values to 1
        mask_skeleton = create_skeleton(mask)  # Pull the base skeleton 
        skeleton_extended = compute_extended_path(mask_skeleton, mask, num_points_src=5, num_points_dst=5)[3]  # Pull extended medial line

        skeleton_resampled = resample_line(skeleton_extended, num_points=20)  # Resample points to smooth
        skeleton_TL = line_length(skeleton_resampled)  # Extract medial TL

        cross_sectional_points = get_cross_sectional_points(skeleton_resampled, mask)  # Extract cross-sectional points
        ls_fs_ps = get_ls_fs_ps(cross_sectional_points, mask)  # Pull out the LS, FS, PS dictionary

        # Verify expected keys
        if "LS" not in ls_fs_ps or "FS" not in ls_fs_ps or "PS" not in ls_fs_ps:
            print(f"Warning: Missing keys in ls_fs_ps for {file}. Current keys: {ls_fs_ps.keys()}")
            continue  # Skip further calculation if keys are missing    

        # Compute body span
        body_span = compute_bodyspan(ls_fs_ps)  # Compute the average body span
        
        image_name = file.replace('_annotation', '').replace('.png', '.JPG')  # Revert image name
        data.append((image_name, skeleton_TL, body_span))  # Append tuple 

    df = pd.DataFrame(data, columns=['filename', 'skeleton_TL', 'body_span'])  # Construct dataframe
        
    return df

In [38]:
df = process_biometrics(root_pngs, annot_files)

KeyboardInterrupt: 

In [ ]:
df